In [4]:
from pathlib import Path
import pandas as pd

project_root = Path.home() / "Documents" / "airline-delay-reliability-analysis"
raw_dir = project_root / "data" / "raw"

csv_files = list(raw_dir.rglob("*.csv"))

csv_files

[WindowsPath('C:/Users/21129/Documents/airline-delay-reliability-analysis/data/raw/On_Time_Marketing_Carrier_On_Tim/On_Time_Marketing_Carrier_On_Time_Performance_(Beginning_January_2018)_2025_1.csv')]

In [5]:
csv_path = csv_files[0]

sample_df = pd.read_csv(
    csv_path,
    nrows=5,
    low_memory=False
)

print(f"Number of columns: {len(sample_df.columns)}")
print(sample_df.columns.tolist())

Number of columns: 120
['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Marketing_Airline_Network', 'Operated_or_Branded_Code_Share_Partners', 'DOT_ID_Marketing_Airline', 'IATA_Code_Marketing_Airline', 'Flight_Number_Marketing_Airline', 'Originally_Scheduled_Code_Share_Airline', 'DOT_ID_Originally_Scheduled_Code_Share_Airline', 'IATA_Code_Originally_Scheduled_Code_Share_Airline', 'Flight_Num_Originally_Scheduled_Code_Share_Airline', 'Operating_Airline ', 'DOT_ID_Operating_Airline', 'IATA_Code_Operating_Airline', 'Tail_Number', 'Flight_Number_Operating_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut

In [6]:
selected_columns = [
    "Year",
    "Month",
    "DayofMonth",
    "DayOfWeek",
    "FlightDate",
    "Marketing_Airline_Network",
    "Operating_Airline",
    "Tail_Number",
    "Flight_Number_Operating_Airline",
    "Origin",
    "OriginCityName",
    "OriginState",
    "Dest",
    "DestCityName",
    "DestState",
    "CRSDepTime",
    "DepTime",
    "DepDelayMinutes",
    "DepDel15",
    "TaxiOut",
    "WheelsOff",
    "WheelsOn",
    "TaxiIn",
    "CRSArrTime",
    "ArrTime",
    "ArrDelayMinutes",
    "ArrDel15",
    "CRSElapsedTime",
    "ActualElapsedTime",
    "AirTime",
    "Cancelled",
    "CancellationCode",
    "Diverted",
    "Distance",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]

missing_columns = [
    col for col in selected_columns
    if col not in sample_df.columns
]

print(f"Selected columns: {len(selected_columns)}")
print(f"Missing columns: {missing_columns}")

Selected columns: 39
Missing columns: ['Operating_Airline']


In [4]:
[col for col in sample_df.columns if "Operating" in col]

['Operating_Airline ',
 'DOT_ID_Operating_Airline',
 'IATA_Code_Operating_Airline',
 'Flight_Number_Operating_Airline']

In [7]:
sample_df.columns = sample_df.columns.str.strip()

missing_columns = [
    col for col in selected_columns
    if col not in sample_df.columns
]

print(f"Selected columns: {len(selected_columns)}")
print(f"Missing columns: {missing_columns}")

Selected columns: 39
Missing columns: []


In [8]:
january_df = pd.read_csv(
    csv_path,
    usecols=lambda col: col.strip() in selected_columns,
    low_memory=False
)

january_df.columns = january_df.columns.str.strip()

rows_before_filter = len(january_df)

january_df = january_df[
    january_df["Marketing_Airline_Network"].isin(["AA", "DL", "UA"])
].copy()

rows_after_filter = len(january_df)
memory_mb = january_df.memory_usage(deep=True).sum() / 1024**2

print(f"Rows before airline filter: {rows_before_filter:,}")
print(f"Rows after AA/DL/UA filter: {rows_after_filter:,}")
print(f"Columns: {january_df.shape[1]}")
print(f"Memory usage: {memory_mb:.2f} MB")
print()
print("Marketing airline counts:")
print(january_df["Marketing_Airline_Network"].value_counts())

Rows before airline filter: 599,013
Rows after AA/DL/UA filter: 394,873
Columns: 39
Memory usage: 142.40 MB

Marketing airline counts:
Marketing_Airline_Network
AA    155779
DL    123672
UA    115422
Name: count, dtype: int64


In [10]:
core_airports = [
    "ATL", "DFW", "DEN", "ORD",
    "CLT", "MIA", "PHX",
    "DTW", "MSP", "SLC",
    "JFK", "LGA",
    "IAH", "EWR", "SFO", "LAX"
]

both_airports_mask = (
    january_df["Origin"].isin(core_airports) &
    january_df["Dest"].isin(core_airports)
)

either_airport_mask = (
    january_df["Origin"].isin(core_airports) |
    january_df["Dest"].isin(core_airports)
)

both_count = both_airports_mask.sum()
either_count = either_airport_mask.sum()

print(f"AA/DL/UA January flights: {len(january_df):,}")
print(f"Both origin and destination in core airports: {both_count:,}")
print(f"Either origin or destination in core airports: {either_count:,}")

print()
print(f"Both-airport coverage: {both_count / len(january_df):.1%}")
print(f"Either-airport coverage: {either_count / len(january_df):.1%}")

AA/DL/UA January flights: 394,873
Both origin and destination in core airports: 63,434
Either origin or destination in core airports: 352,729

Both-airport coverage: 16.1%
Either-airport coverage: 89.3%


In [11]:
january_scope_df = january_df[
    january_df["Origin"].isin(core_airports) |
    january_df["Dest"].isin(core_airports)
].copy()

memory_mb = january_scope_df.memory_usage(deep=True).sum() / 1024**2

print(f"Rows: {len(january_scope_df):,}")
print(f"Columns: {january_scope_df.shape[1]}")
print(f"Memory usage: {memory_mb:.2f} MB")
print(f"Unique origins: {january_scope_df['Origin'].nunique()}")
print(f"Unique destinations: {january_scope_df['Dest'].nunique()}")

Rows: 352,729
Columns: 39
Memory usage: 127.18 MB
Unique origins: 300
Unique destinations: 300


In [12]:
january_scope_df["FlightDate"] = pd.to_datetime(
    january_scope_df["FlightDate"]
)

quality_columns = [
    "FlightDate",
    "Marketing_Airline_Network",
    "Operating_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
    "CRSArrTime",
    "ArrDelayMinutes",
    "ArrDel15",
    "Cancelled",
    "Diverted",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]

print("Date range:")
print(
    january_scope_df["FlightDate"].min(),
    "to",
    january_scope_df["FlightDate"].max()
)

print("\nCancelled:")
print(january_scope_df["Cancelled"].value_counts(dropna=False))

print("\nDiverted:")
print(january_scope_df["Diverted"].value_counts(dropna=False))

print("\nArrDel15:")
print(january_scope_df["ArrDel15"].value_counts(dropna=False))

print("\nMissing values:")
print(
    january_scope_df[quality_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

Date range:
2025-01-01 00:00:00 to 2025-01-31 00:00:00

Cancelled:
Cancelled
0.0    341189
1.0     11540
Name: count, dtype: int64

Diverted:
Diverted
0.0    351962
1.0       767
Name: count, dtype: int64

ArrDel15:
ArrDel15
0.0    275217
1.0     65205
NaN     12307
Name: count, dtype: int64

Missing values:
WeatherDelay                 287524
NASDelay                     287524
SecurityDelay                287524
LateAircraftDelay            287524
CarrierDelay                 287524
ArrDelayMinutes               12307
ArrDel15                      12307
FlightDate                        0
CRSArrTime                        0
CRSDepTime                        0
Dest                              0
Origin                            0
Marketing_Airline_Network         0
Operating_Airline                 0
Diverted                          0
Cancelled                         0
dtype: int64


In [13]:
zip_files = sorted(raw_dir.glob("*.zip"))

print(f"Number of ZIP files: {len(zip_files)}")

for file in zip_files:
    print(file.name)

Number of ZIP files: 12
On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2025_1.zip
On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2025_10.zip
On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2025_11.zip
On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2025_12.zip
On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2025_2.zip
On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2025_3.zip
On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2025_4.zip
On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2025_5.zip
On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2025_6.zip
On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2025_7.zip
On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2025_8.zip
On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2025_9.zip


In [14]:
import zipfile

def process_monthly_zip(zip_path, selected_columns, core_airports):
    with zipfile.ZipFile(zip_path, "r") as zip_file:
        csv_names = [
            name for name in zip_file.namelist()
            if name.lower().endswith(".csv")
        ]

        if len(csv_names) != 1:
            raise ValueError(
                f"Expected 1 CSV in {zip_path.name}, found {len(csv_names)}"
            )

        with zip_file.open(csv_names[0]) as csv_file:
            df = pd.read_csv(
                csv_file,
                usecols=lambda col: col.strip() in selected_columns,
                low_memory=False
            )

    df.columns = df.columns.str.strip()

    df = df[
        df["Marketing_Airline_Network"].isin(["AA", "DL", "UA"])
    ].copy()

    df = df[
        df["Origin"].isin(core_airports) |
        df["Dest"].isin(core_airports)
    ].copy()

    df["FlightDate"] = pd.to_datetime(df["FlightDate"])

    return df

In [15]:
feb_zip = [
    file for file in zip_files
    if file.name.endswith("_2025_2.zip")
][0]

february_df = process_monthly_zip(
    feb_zip,
    selected_columns,
    core_airports
)

print(f"File: {feb_zip.name}")
print(f"Rows: {len(february_df):,}")
print(f"Columns: {february_df.shape[1]}")
print(
    f"Date range: {february_df['FlightDate'].min()} "
    f"to {february_df['FlightDate'].max()}"
)
print("\nAirline counts:")
print(february_df["Marketing_Airline_Network"].value_counts())

File: On_Time_Marketing_Carrier_On_Time_Performance_Beginning_January_2018_2025_2.zip
Rows: 326,439
Columns: 39
Date range: 2025-02-01 00:00:00 to 2025-02-28 00:00:00

Airline counts:
Marketing_Airline_Network
AA    124723
DL    103975
UA     97741
Name: count, dtype: int64


In [16]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
feb_output = processed_monthly_dir / "flights_2025_02.parquet"

february_df.to_parquet(
    feb_output,
    index=False
)

file_size_mb = feb_output.stat().st_size / 1024**2

print(f"Saved to: {feb_output}")
print(f"File size: {file_size_mb:.2f} MB")

NameError: name 'processed_monthly_dir' is not defined

In [18]:
import pandas as pd
import pyarrow as pa

print(f"pandas version: {pd.__version__}")
print(f"pyarrow version: {pa.__version__}")

pandas version: 3.0.3
pyarrow version: 25.0.1


In [19]:
processed_monthly_dir = project_root / "data" / "processed" / "monthly"
processed_monthly_dir.mkdir(parents=True, exist_ok=True)

print(processed_monthly_dir)

C:\Users\21129\Documents\airline-delay-reliability-analysis\data\processed\monthly


In [20]:
feb_output = processed_monthly_dir / "flights_2025_02.parquet"

february_df.to_parquet(
    feb_output,
    index=False,
    engine="pyarrow"
)

file_size_mb = feb_output.stat().st_size / 1024**2

print(f"Saved to: {feb_output}")
print(f"File size: {file_size_mb:.2f} MB")

Saved to: C:\Users\21129\Documents\airline-delay-reliability-analysis\data\processed\monthly\flights_2025_02.parquet
File size: 6.81 MB


In [21]:
february_check = pd.read_parquet(
    feb_output,
    engine="pyarrow"
)

print(f"Rows: {february_check.shape[0]:,}")
print(f"Columns: {february_check.shape[1]}")
print(
    f"Date range: {february_check['FlightDate'].min()} "
    f"to {february_check['FlightDate'].max()}"
)
print(
    "Airlines:",
    sorted(february_check["Marketing_Airline_Network"].unique())
)

Rows: 326,439
Columns: 39
Date range: 2025-02-01 00:00:00 to 2025-02-28 00:00:00
Airlines: ['AA', 'DL', 'UA']


In [22]:
import re
import gc

zip_files = list(raw_dir.glob("*.zip"))

def get_month_from_filename(file_path):
    match = re.search(r"_2025_(\d+)\.zip$", file_path.name)

    if match is None:
        raise ValueError(f"Cannot identify month from: {file_path.name}")

    return int(match.group(1))


zip_files = sorted(
    zip_files,
    key=get_month_from_filename
)

monthly_summary = []

for zip_path in zip_files:
    month = get_month_from_filename(zip_path)

    print(f"Processing 2025-{month:02d}...")

    monthly_df = process_monthly_zip(
        zip_path,
        selected_columns,
        core_airports
    )

    output_path = (
        processed_monthly_dir /
        f"flights_2025_{month:02d}.parquet"
    )

    monthly_df.to_parquet(
        output_path,
        index=False,
        engine="pyarrow"
    )

    monthly_summary.append({
        "month": month,
        "rows": len(monthly_df),
        "columns": monthly_df.shape[1],
        "file_size_mb": output_path.stat().st_size / 1024**2
    })

    print(
        f"Saved 2025-{month:02d}: "
        f"{len(monthly_df):,} rows"
    )

    del monthly_df
    gc.collect()

print("\nAll months processed successfully.")

Processing 2025-01...
Saved 2025-01: 352,729 rows
Processing 2025-02...
Saved 2025-02: 326,439 rows
Processing 2025-03...
Saved 2025-03: 382,196 rows
Processing 2025-04...
Saved 2025-04: 370,593 rows
Processing 2025-05...
Saved 2025-05: 387,942 rows
Processing 2025-06...
Saved 2025-06: 389,996 rows
Processing 2025-07...
Saved 2025-07: 403,607 rows
Processing 2025-08...
Saved 2025-08: 390,303 rows
Processing 2025-09...
Saved 2025-09: 369,158 rows
Processing 2025-10...
Saved 2025-10: 394,814 rows
Processing 2025-11...
Saved 2025-11: 372,492 rows
Processing 2025-12...
Saved 2025-12: 378,724 rows

All months processed successfully.


In [23]:
monthly_summary_df = pd.DataFrame(monthly_summary)

monthly_summary_df["file_size_mb"] = (
    monthly_summary_df["file_size_mb"].round(2)
)

print(monthly_summary_df)

print("\nAnnual summary:")
print(f"Months: {len(monthly_summary_df)}")
print(f"Total rows: {monthly_summary_df['rows'].sum():,}")
print(f"Minimum monthly rows: {monthly_summary_df['rows'].min():,}")
print(f"Maximum monthly rows: {monthly_summary_df['rows'].max():,}")

    month    rows  columns  file_size_mb
0       1  352729       39          7.65
1       2  326439       39          6.81
2       3  382196       39          8.23
3       4  370593       39          7.56
4       5  387942       39          8.13
5       6  389996       39          8.57
6       7  403607       39          8.70
7       8  390303       39          8.36
8       9  369158       39          7.78
9      10  394814       39          8.36
10     11  372492       39          7.84
11     12  378724       39          8.49

Annual summary:
Months: 12
Total rows: 4,518,993
Minimum monthly rows: 326,439
Maximum monthly rows: 403,607


In [24]:
data_dictionary = pd.DataFrame([
    ["Year", "Calendar year of the flight", "Time"],
    ["Month", "Calendar month of the flight", "Time"],
    ["DayofMonth", "Day of month", "Time"],
    ["DayOfWeek", "Day of week coded by BTS", "Time"],
    ["FlightDate", "Scheduled flight date", "Time"],

    ["Marketing_Airline_Network", "Marketing carrier network code", "Airline"],
    ["Operating_Airline", "Airline operating the flight", "Airline"],
    ["Tail_Number", "Aircraft tail number", "Aircraft"],
    ["Flight_Number_Operating_Airline", "Operating carrier flight number", "Flight"],

    ["Origin", "Origin airport IATA code", "Network"],
    ["OriginCityName", "Origin city name", "Network"],
    ["OriginState", "Origin state code", "Network"],
    ["Dest", "Destination airport IATA code", "Network"],
    ["DestCityName", "Destination city name", "Network"],
    ["DestState", "Destination state code", "Network"],
    ["Distance", "Flight distance in miles", "Route"],

    ["CRSDepTime", "Scheduled departure time", "Schedule"],
    ["CRSArrTime", "Scheduled arrival time", "Schedule"],
    ["CRSElapsedTime", "Scheduled elapsed flight time in minutes", "Schedule"],

    ["DepTime", "Actual departure time", "Operations"],
    ["DepDelayMinutes", "Departure delay in minutes, non-negative", "Operations"],
    ["DepDel15", "Indicator for departure delay of at least 15 minutes", "Operations"],
    ["TaxiOut", "Taxi-out time in minutes", "Operations"],
    ["WheelsOff", "Time aircraft became airborne", "Operations"],
    ["WheelsOn", "Time aircraft landed", "Operations"],
    ["TaxiIn", "Taxi-in time in minutes", "Operations"],
    ["ArrTime", "Actual arrival time", "Operations"],
    ["ArrDelayMinutes", "Arrival delay in minutes, non-negative", "Operations"],
    ["ArrDel15", "Indicator for arrival delay of at least 15 minutes", "Target"],
    ["ActualElapsedTime", "Actual elapsed flight time in minutes", "Operations"],
    ["AirTime", "Time spent airborne in minutes", "Operations"],

    ["Cancelled", "Indicator showing whether the flight was cancelled", "Irregular Operations"],
    ["CancellationCode", "BTS cancellation reason code", "Irregular Operations"],
    ["Diverted", "Indicator showing whether the flight was diverted", "Irregular Operations"],

    ["CarrierDelay", "Delay minutes attributed to the air carrier", "Delay Cause"],
    ["WeatherDelay", "Delay minutes attributed to weather", "Delay Cause"],
    ["NASDelay", "Delay minutes attributed to the National Aviation System", "Delay Cause"],
    ["SecurityDelay", "Delay minutes attributed to security", "Delay Cause"],
    ["LateAircraftDelay", "Delay minutes attributed to late-arriving aircraft", "Delay Cause"]
], columns=[
    "field_name",
    "description",
    "category"
])

print(f"Fields in data dictionary: {len(data_dictionary)}")
data_dictionary

Fields in data dictionary: 39


,field_name,description,category
0,Year,Calendar year of the flight,Time
1,Month,Calendar month of the flight,Time
2,DayofMonth,Day of month,Time
3,DayOfWeek,Day of week coded by BTS,Time
4,FlightDate,Scheduled flight date,Time
5,Marketing_Airline_Network,Marketing carrier network code,Airline
6,Operating_Airline,Airline operating the flight,Airline
7,Tail_Number,Aircraft tail number,Aircraft
8,Flight_Number_Operating_Airline,Operating carrier flight number,Flight
9,Origin,Origin airport IATA code,Network


In [25]:
docs_dir = project_root / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

dictionary_path = docs_dir / "data_dictionary.csv"

data_dictionary.to_csv(
    dictionary_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved to: {dictionary_path}")
print(f"Rows: {len(data_dictionary)}")

Saved to: C:\Users\21129\Documents\airline-delay-reliability-analysis\docs\data_dictionary.csv
Rows: 39


In [1]:
%pip install sqlalchemy pymysql

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -------- ------------------------------- 0.5/2.4 MB 4.4 MB/s eta 0:00:01
   -------------------------- ------------- 1.6/2.4 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 4.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sqlalchemy
import pymysql

print(f"SQLAlchemy version: {sqlalchemy.__version__}")
print(f"PyMySQL version: {pymysql.__version__}")

SQLAlchemy version: 2.1.1
PyMySQL version: 2.2.8


In [4]:
from getpass import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

mysql_user = "root"
mysql_password = getpass("Enter new MySQL password: ")

connection_url = URL.create(
    "mysql+pymysql",
    username=mysql_user,
    password=mysql_password,
    host="localhost",
    port=3306,
    database="airline_reliability"
)

engine = create_engine(
    connection_url,
    pool_pre_ping=True
)

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT DATABASE() AS database_name, 1 AS connection_test;")
    )
    print(result.fetchone())

Enter new MySQL password:  ········


('airline_reliability', 1)


In [5]:
from pathlib import Path
import pandas as pd

project_root = Path.home() / "Documents" / "airline-delay-reliability-analysis"

feb_path = (
    project_root
    / "data"
    / "processed"
    / "monthly"
    / "flights_2025_02.parquet"
)

february_df = pd.read_parquet(
    feb_path,
    engine="pyarrow"
)

column_mapping = {
    "Year": "flight_year",
    "Month": "flight_month",
    "DayofMonth": "day_of_month",
    "DayOfWeek": "day_of_week",
    "FlightDate": "flight_date",
    "Marketing_Airline_Network": "marketing_airline_network",
    "Operating_Airline": "operating_airline",
    "Tail_Number": "tail_number",
    "Flight_Number_Operating_Airline": "flight_number_operating_airline",
    "Origin": "origin",
    "OriginCityName": "origin_city_name",
    "OriginState": "origin_state",
    "Dest": "dest",
    "DestCityName": "dest_city_name",
    "DestState": "dest_state",
    "Distance": "distance",
    "CRSDepTime": "crs_dep_time",
    "DepTime": "dep_time",
    "DepDelayMinutes": "dep_delay_minutes",
    "DepDel15": "dep_del15",
    "TaxiOut": "taxi_out",
    "WheelsOff": "wheels_off",
    "WheelsOn": "wheels_on",
    "TaxiIn": "taxi_in",
    "CRSArrTime": "crs_arr_time",
    "ArrTime": "arr_time",
    "ArrDelayMinutes": "arr_delay_minutes",
    "ArrDel15": "arr_del15",
    "CRSElapsedTime": "crs_elapsed_time",
    "ActualElapsedTime": "actual_elapsed_time",
    "AirTime": "air_time",
    "Cancelled": "cancelled",
    "CancellationCode": "cancellation_code",
    "Diverted": "diverted",
    "CarrierDelay": "carrier_delay",
    "WeatherDelay": "weather_delay",
    "NASDelay": "nas_delay",
    "SecurityDelay": "security_delay",
    "LateAircraftDelay": "late_aircraft_delay"
}

february_sql_df = february_df.rename(columns=column_mapping)

print(f"Rows: {february_sql_df.shape[0]:,}")
print(f"Columns: {february_sql_df.shape[1]}")
print(f"Renamed columns: {len(column_mapping)}")
print()
print(february_sql_df.columns.tolist())

Rows: 326,439
Columns: 39
Renamed columns: 39

['flight_year', 'flight_month', 'day_of_month', 'day_of_week', 'flight_date', 'marketing_airline_network', 'operating_airline', 'tail_number', 'flight_number_operating_airline', 'origin', 'origin_city_name', 'origin_state', 'dest', 'dest_city_name', 'dest_state', 'crs_dep_time', 'dep_time', 'dep_delay_minutes', 'dep_del15', 'taxi_out', 'wheels_off', 'wheels_on', 'taxi_in', 'crs_arr_time', 'arr_time', 'arr_delay_minutes', 'arr_del15', 'cancelled', 'cancellation_code', 'diverted', 'crs_elapsed_time', 'actual_elapsed_time', 'air_time', 'distance', 'carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']


In [6]:
test_df = february_sql_df.head(1000).copy()

test_df.to_sql(
    name="flights_2025_test",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=200
)

print(f"Rows sent to MySQL: {len(test_df):,}")

Rows sent to MySQL: 1,000


In [8]:
jan_path = (
    project_root
    / "data"
    / "processed"
    / "monthly"
    / "flights_2025_01.parquet"
)

january_sql_df = pd.read_parquet(
    jan_path,
    engine="pyarrow"
).rename(columns=column_mapping)

print(f"January rows ready: {len(january_sql_df):,}")

January rows ready: 352,729


In [10]:
import time

start_time = time.time()

with engine.begin() as connection:
    january_sql_df.to_sql(
        name="flights_2025",
        con=connection,
        if_exists="append",
        index=False,
        chunksize=2000
    )

elapsed_minutes = (time.time() - start_time) / 60

print(f"January rows inserted: {len(january_sql_df):,}")
print(f"Elapsed time: {elapsed_minutes:.2f} minutes")

January rows inserted: 352,729
Elapsed time: 0.49 minutes


In [11]:
from sqlalchemy import text
import time

month = 1

start_time = time.time()

with engine.begin() as connection:
    
    connection.execute(
        text("""
            DELETE FROM flights_2025
            WHERE flight_year = 2025
              AND flight_month = :month
        """),
        {"month": month}
    )

    january_sql_df.to_sql(
        name="flights_2025",
        con=connection,
        if_exists="append",
        index=False,
        chunksize=2000
    )

elapsed_minutes = (time.time() - start_time) / 60

with engine.connect() as connection:
    january_count = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM flights_2025
            WHERE flight_year = 2025
              AND flight_month = 1
        """)
    ).scalar()

print(f"January rows in MySQL: {january_count:,}")
print(f"Elapsed time: {elapsed_minutes:.2f} minutes")

January rows in MySQL: 352,729
Elapsed time: 0.46 minutes


In [12]:
import gc
import time
import pandas as pd
from sqlalchemy import text

import_log = []

for month in range(2, 13):
    print(f"\nProcessing 2025-{month:02d}...")

    monthly_path = (
        project_root
        / "data"
        / "processed"
        / "monthly"
        / f"flights_2025_{month:02d}.parquet"
    )

    monthly_df = pd.read_parquet(
        monthly_path,
        engine="pyarrow"
    ).rename(columns=column_mapping)

    expected_rows = len(monthly_df)

    start_time = time.time()

    with engine.begin() as connection:

        connection.execute(
            text("""
                DELETE FROM flights_2025
                WHERE flight_year = 2025
                  AND flight_month = :month
            """),
            {"month": month}
        )

        monthly_df.to_sql(
            name="flights_2025",
            con=connection,
            if_exists="append",
            index=False,
            chunksize=2000
        )

    with engine.connect() as connection:
        mysql_rows = connection.execute(
            text("""
                SELECT COUNT(*)
                FROM flights_2025
                WHERE flight_year = 2025
                  AND flight_month = :month
            """),
            {"month": month}
        ).scalar()

    elapsed_minutes = (time.time() - start_time) / 60

    status = "OK" if mysql_rows == expected_rows else "CHECK"

    import_log.append({
        "month": month,
        "expected_rows": expected_rows,
        "mysql_rows": mysql_rows,
        "status": status,
        "minutes": round(elapsed_minutes, 2)
    })

    print(f"Expected rows: {expected_rows:,}")
    print(f"MySQL rows:    {mysql_rows:,}")
    print(f"Status:        {status}")
    print(f"Time:          {elapsed_minutes:.2f} minutes")

    del monthly_df
    gc.collect()

print("\nFebruary–December import finished.")


Processing 2025-02...
Expected rows: 326,439
MySQL rows:    326,439
Status:        OK
Time:          0.41 minutes

Processing 2025-03...
Expected rows: 382,196
MySQL rows:    382,196
Status:        OK
Time:          0.50 minutes

Processing 2025-04...
Expected rows: 370,593
MySQL rows:    370,593
Status:        OK
Time:          0.55 minutes

Processing 2025-05...
Expected rows: 387,942
MySQL rows:    387,942
Status:        OK
Time:          0.60 minutes

Processing 2025-06...
Expected rows: 389,996
MySQL rows:    389,996
Status:        OK
Time:          0.65 minutes

Processing 2025-07...
Expected rows: 403,607
MySQL rows:    403,607
Status:        OK
Time:          0.72 minutes

Processing 2025-08...
Expected rows: 390,303
MySQL rows:    390,303
Status:        OK
Time:          0.76 minutes

Processing 2025-09...
Expected rows: 369,158
MySQL rows:    369,158
Status:        OK
Time:          0.74 minutes

Processing 2025-10...
Expected rows: 394,814
MySQL rows:    394,814
Status:    